In [0]:
from pyspark.sql.functions import *
from pyspark.ml.feature import Tokenizer
from pyspark.ml.feature import StopWordsRemover
from pyspark.ml.feature import HashingTF
from pyspark.ml.feature import IDF
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

print("=" * 60)
print("REAL NLP HEALTHCARE INSIGHTS")
print("=" * 60)

# ============================================
# CREATE SAMPLE CLINICAL NOTES
# ============================================

clinical_notes = [

    (1, "Patient has diabetes and high blood pressure"),
    (2, "Heart disease symptoms observed in patient"),
    (3, "Patient diagnosed with pneumonia infection"),
    (4, "Diabetes treatment improved patient condition"),
    (5, "Severe chest pain and heart complications"),
    (6, "Lung infection and breathing difficulty"),
    (7, "Patient has hypertension and diabetes"),
    (8, "Cardiac arrest symptoms identified"),
    (9, "Respiratory infection observed"),
    (10, "Blood sugar levels are elevated")

]

notes_df = spark.createDataFrame(
    clinical_notes,
    ["note_id", "clinical_note"]
)

display(notes_df)

# ============================================
# TOKENIZATION
# ============================================

tokenizer = Tokenizer(
    inputCol="clinical_note",
    outputCol="tokens"
)

# ============================================
# REMOVE STOP WORDS
# ============================================

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

# ============================================
# TEXT VECTORIZATION
# ============================================

hashingTF = HashingTF(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    numFeatures=20
)

# ============================================
# TF-IDF
# ============================================

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

# ============================================
# NLP CLUSTERING MODEL
# ============================================

kmeans = KMeans(
    featuresCol="features",
    predictionCol="topic_cluster",
    k=3,
    seed=42
)

# ============================================
# NLP PIPELINE
# ============================================

pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    hashingTF,
    idf,
    kmeans
])

# ============================================
# TRAIN NLP MODEL
# ============================================

model = pipeline.fit(notes_df)

print("NLP Model Trained Successfully")

# ============================================
# GENERATE INSIGHTS
# ============================================

results = model.transform(notes_df)

display(
    results.select(
        "note_id",
        "clinical_note",
        "filtered_tokens",
        "topic_cluster"
    )
)

# ============================================
# SAVE NLP OUTPUT
# ============================================

results.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_migration.gold.real_nlp_healthcare_insights")

print("NLP insights table saved successfully")

print("=" * 60)
print("REAL NLP HEALTHCARE COMPLETED")
print("=" * 60)

REAL NLP HEALTHCARE INSIGHTS


note_id,clinical_note
1,Patient has diabetes and high blood pressure
2,Heart disease symptoms observed in patient
3,Patient diagnosed with pneumonia infection
4,Diabetes treatment improved patient condition
5,Severe chest pain and heart complications
6,Lung infection and breathing difficulty
7,Patient has hypertension and diabetes
8,Cardiac arrest symptoms identified
9,Respiratory infection observed
10,Blood sugar levels are elevated


NLP Model Trained Successfully


note_id,clinical_note,filtered_tokens,topic_cluster
1,Patient has diabetes and high blood pressure,"List(patient, diabetes, high, blood, pressure)",0
2,Heart disease symptoms observed in patient,"List(heart, disease, symptoms, observed, patient)",0
3,Patient diagnosed with pneumonia infection,"List(patient, diagnosed, pneumonia, infection)",1
4,Diabetes treatment improved patient condition,"List(diabetes, treatment, improved, patient, condition)",0
5,Severe chest pain and heart complications,"List(severe, chest, pain, heart, complications)",2
6,Lung infection and breathing difficulty,"List(lung, infection, breathing, difficulty)",0
7,Patient has hypertension and diabetes,"List(patient, hypertension, diabetes)",0
8,Cardiac arrest symptoms identified,"List(cardiac, arrest, symptoms, identified)",0
9,Respiratory infection observed,"List(respiratory, infection, observed)",0
10,Blood sugar levels are elevated,"List(blood, sugar, levels, elevated)",0


NLP insights table saved successfully
REAL NLP HEALTHCARE COMPLETED
